# nano-dsv4.1f — Kaggle TPU v5e-8 operator validation

Select TPU v5e-8 and Internet. Start a fresh session and run all cells. The bootstrap pins JAX/jaxlib/libtpu before any JAX import. Set `NANO_DSV41F_REF` to a branch or commit before bootstrap to test it; default is `main`. Training runs in a fresh Python process. Results are written under `/kaggle/working`. This synthetic test is not the official pretraining data/recipe.

In [ ]:
"""Canonical notebook bootstrap. Standard library only: runs before importing JAX."""
from pathlib import Path
from importlib.metadata import version
import os
import subprocess
import sys

if any(name in sys.modules for name in ("jax", "jaxlib", "libtpu", "nano_dsv41f")):
    raise RuntimeError("Restart the Kaggle session and run this cell first: JAX/TPU was already imported.")

ROOT = Path('/kaggle/working/nano-dsv4.1f')
REPO_REF = os.environ.get('NANO_DSV41F_REF', 'main')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(ROOT)], check=True)
if not (ROOT / '.git').exists():
    raise RuntimeError(f'{ROOT} exists but is not a git checkout.')
status = subprocess.check_output(['git', '-C', str(ROOT), 'status', '--porcelain',
                                  '--untracked-files=no'], text=True)
if status.strip():
    raise RuntimeError('Checkout has tracked edits; preserve them before updating:\n' + status)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
# Pin the entire validated package combination, not just the Python jax package.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                '-r', str(ROOT / 'requirements-tpu.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT), '--no-deps'], check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['JAX_PLATFORMS'] = 'tpu'
print('repo:', ROOT)
print('requested ref:', REPO_REF)
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('packages:', {p: version(p) for p in ('jax', 'jaxlib', 'libtpu')})
# Execution cells run fresh Python processes. Never retain an old TPU client in IPython.


## Packed, multi-expert and late-indexer smoke

One Pallas preflight, then 10 base + 10 late steps. Set the query budget below: it caps eligible positions across the batch per retriever. L5 training uses full legal history by default. Report includes synchronized timing, compiler memory, collectives, query coverage and loss trajectories.

In [ ]:
QUERY_BUDGET = 128
QUERY_SEED = 0
APPLY_CANDIDATE_MASK = False
command = [sys.executable, '-u', 'scripts/run_combined_tpu_smoke.py',
           '--query-budget', str(QUERY_BUDGET), '--query-seed', str(QUERY_SEED),
           '--output', '/kaggle/working/combined-smoke.json']
if APPLY_CANDIDATE_MASK:
    command.append('--apply-candidate-mask')
subprocess.run(command, check=True)